<a href="https://colab.research.google.com/github/Sravani-939/genai-training-tasks/blob/main/Rag_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install chromadb sentence-transformers transformers openai

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"]=userdata.get('OPENAI_API_KEY')

In [ ]:

import uuid
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
from openai import OpenAI

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

#Chromadb setup
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="rag_collection")


def chunk_text(text, chunk_size=200, overlap=50):
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    start = 0

    while start < len(tokens):
        end = start + chunk_size
        chunk_tokens = tokens[start:end]
        chunk = tokenizer.decode(chunk_tokens)

        if chunk.strip():
            chunks.append(chunk)

        if end >= len(tokens):
            break

        start = start + chunk_size - overlap

    return chunks

#storing chunks in chromadb
def store_documents(documents):
    all_chunks = []
    all_ids = []

    for doc in documents:
        chunks = chunk_text(doc, chunk_size=200, overlap=50)
        for chunk in chunks:
            all_chunks.append(chunk)
            all_ids.append(str(uuid.uuid4()))

    embeddings = embedding_model.encode(all_chunks).tolist()

    collection.add(
        ids=all_ids,
        documents=all_chunks,
        embeddings=embeddings
    )


def retrieve(query, top_k=3):
    query_embedding = embedding_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    return results["documents"][0]


def build_prompt(question, passages):
    context = "\n\n".join(passages)

    prompt = f"""
Use only this context to answer the question.
If the answer is not available in the context, say:
"I could not find the answer in the provided context."

Context:
{context}

Question:
{question}

Answer:
"""
    return prompt

def ask_llm(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer only from the given context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content


def rag_pipeline(question, top_k=3):
    top_chunks = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, top_chunks)
    answer = ask_llm(prompt)
    return answer


documents = [
    """Sliding window chunking splits text into overlapping chunks.
    This helps keep context from being lost between chunks.""",

    """Recursive chunking first tries to split text by paragraphs,
    then by sentences, and finally by tokens if needed.""",

    """Vector databases store text embeddings for semantic search.
    Examples include FAISS, Chroma, Milvus, and Weaviate.""",

    """Top-k retrieval means returning the most relevant k chunks
    for a user query based on similarity search.""",

    """RAG stands for Retrieval-Augmented Generation.
    It retrieves relevant passages and gives them to the LLM before answering."""
]

store_documents(documents)

test_questions = [
    "What is sliding window chunking?",
    "What is recursive chunking?",
    "What is a vector database?",
    "What does top-k retrieval mean?",
    "What is RAG?"
]

#results
for question in test_questions:
    print("\n" + " " * 60)
    print("Question:", question)

    retrieved_chunks = retrieve(question, top_k=3)
    print("\nRetrieved Chunks:")
    for i, chunk in enumerate(retrieved_chunks, 1):
        print(f"{i}. {chunk}")

    answer = rag_pipeline(question, top_k=3)
    print("\nFinal Answer:")
    print(answer)